# NEM battery trading backtest
Explore NSW 5-minute NEM prices, fit a simple price-spike model and compare battery trading rules.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from nem_trading.features import build_features
from nem_trading.model import fit_spike_model
from nem_trading.backtest import run_backtest, summarise_backtest
from nem_trading.strategies import threshold_signal, time_aware_signal, probability_aware_signal

## 1. Load and inspect NSW prices

In [ ]:
prices = pd.read_csv('../data/processed/nsw1_prices.csv', parse_dates=['SETTLEMENTDATE'])
prices.describe()

In [ ]:
prices.plot(x='SETTLEMENTDATE', y='RRP', figsize=(12,4), title='NSW NEM spot price')
plt.ylabel('RRP ($/MWh)');

## 2. Create a next-30-minute spike target
The target is 1 when any of the next six 5-minute prices is at least $150/MWh. Features use only information known at the current interval.

In [ ]:
features = build_features(prices, spike_threshold=150)
features[['SETTLEMENTDATE','RRP','TOTALDEMAND','price_change_30m','hour','spike_next_30m']].head()

## 3. Fit an interpretable logistic-regression baseline
The data is split chronologically: the first 70% is training data and the final 30% is held out for testing.

In [ ]:
model_result = fit_spike_model(features)
model_result.metrics

## 4. Compare battery strategies

In [ ]:
test = model_result.test
results = {
    'Threshold': run_backtest(test, threshold_signal),
    'Time-aware': run_backtest(test, time_aware_signal),
    'Probability-aware': run_backtest(test, probability_aware_signal),
}
pd.DataFrame({name: summarise_backtest(result) for name, result in results.items()}).T

In [ ]:
ax = None
for name, result in results.items():
    ax = result.plot(x='SETTLEMENTDATE', y='cumulative_pnl', label=name, ax=ax, figsize=(12,4))
ax.set_ylabel('Cumulative P&L ($)');

## 5. Sensitivity checks
Useful next checks are changing the charge/discharge thresholds, starting state of charge, battery power and efficiency, then comparing how robust the P&L is across different months.